In [14]:
!pip install ultralytics

                                              0.0/904.3 kB ? eta -:--:--
     -----------------                      419.8/904.3 kB 8.7 MB/s eta 0:00:01
     -------------------------------------- 904.3/904.3 kB 9.5 MB/s eta 0:00:00
                                              0.0/8.0 MB ? eta -:--:--
     ----                                     0.9/8.0 MB 19.6 MB/s eta 0:00:01
     ----------                               2.1/8.0 MB 22.4 MB/s eta 0:00:01
     ------------------                       3.7/8.0 MB 26.3 MB/s eta 0:00:01
     -----------------------------            5.9/8.0 MB 31.4 MB/s eta 0:00:01
     ---------------------------------------  8.0/8.0 MB 34.1 MB/s eta 0:00:01
     ---------------------------------------  8.0/8.0 MB 34.1 MB/s eta 0:00:01
     ---------------------------------------- 8.0/8.0 MB 25.6 MB/s eta 0:00:00
                                              0.0/2.6 MB ? eta -:--:--
     -----------------------------------      2.3/2.6 MB 49.2 MB/s eta 0

ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'C:\\Users\\tonys\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\networkx\\algorithms\\flow\\tests\\gl1.gpickle.bz2'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.1.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install packages due to an OSError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'c:\\Users\\tonys\\AppData\\Local\\Programs\\Python\\Python311\\Scripts\\fonttools.exe' -> 'c:\\Users\\tonys\\AppData\\Local\\Programs\\Python\\Python311\\Scripts\\fonttools.exe.deleteme'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.1.2 -> 24.3.1
[notice] To update, run: python.exe -m 

  Using cached ultralytics-8.3.55-py3-none-any.whl (904 kB)
  Using cached matplotlib-3.10.0-cp311-cp311-win_amd64.whl (8.0 MB)
  Using cached pillow-11.0.0-cp311-cp311-win_amd64.whl (2.6 MB)
  Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Using cached torch-2.5.1-cp311-cp311-win_amd64.whl (203.1 MB)
  Using cached torchvision-0.20.1-cp311-cp311-win_amd64.whl (1.6 MB)
  Using cached pandas-2.2.3-cp311-cp311-win_amd64.whl (11.6 MB)
  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
  Using cached ultralytics_thop-2.0.13-py3-none-any.whl (26 kB)
  Using cached contourpy-1.3.1-cp311-cp311-win_amd64.whl (219 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.55.3-cp311-cp311-win_amd64.whl (2.2 MB)
  Using cached kiwisolver-1.4.8-cp311-cp311-win_amd64.whl (71 kB)
  Using cached pyparsing-3.2.0-py3-none-any.whl (106 kB)
  Using cached charset_normalizer-3.4.1-cp311-cp311-win_amd64.whl (102 kB)
  Using cached idna-3.10-py3-none-any.whl (70 kB)


[notice] A new release of pip is available: 23.1.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import os
import numpy as np


In [41]:
from ultralytics import YOLO

# Charger le modèle YOLO
model = YOLO("yolov8s.pt")

# Variables globales pour stocker les coordonnées et vitesses des voitures
car_positions = {}  # {car_id: (x, y)}
car_speeds = {}     # {car_id: speed}
fps = 30  # Fréquence d'images de la vidéo (ajustez selon la vidéo)

def car_detection(frame):
    global car_positions, car_speeds  # Utilisation des variables globales
    results = model(frame)  # Obtenir les résultats de YOLO
    detections = results[0].boxes  # Accéder aux boîtes englobantes

    new_car_positions = {}

    # Parcourir les détections
    for box in detections:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Coordonnées de la boîte
        conf = box.conf[0].item()  # Confiance de la détection
        class_id = int(box.cls[0])  # Classe de l'objet détecté

        if class_id in [2, 3, 5, 7]:  # Filtrer pour voitures/camions uniquement
            x_center = int((x1 + x2) / 2)
            y_center = int((y1 + y2) / 2)

            # Associer un ID à chaque voiture détectée
            car_id = len(new_car_positions)  # Simple incrémentation (peut être améliorée)
            new_car_positions[car_id] = (x_center, y_center)
            
            # Calculer la vitesse si une position précédente existe pour ce car_id
            if car_id in car_positions:
                old_x, old_y = car_positions[car_id]
                dx = x_center - old_x
                dy = y_center - old_y
                distance = ((dx ** 2) + (dy ** 2)) ** 0.5  # Distance euclidienne
                speed_pixels_per_second = distance * fps
                car_speeds[car_id] = speed_pixels_per_second  # Stocker la vitesse

            # Dessiner la voiture et sa vitesse sur l'image
            cv2.circle(frame, (x_center, y_center), 5, (0, 255, 0), -1)  # Point au centre de la voiture
            if car_id in car_speeds:
                speed_text = f"{car_speeds[car_id]:.2f} px/s"
                cv2.putText(frame, speed_text, (x_center, y_center - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # Mettre à jour les positions des voitures pour le prochain frame
    car_positions = new_car_positions
    return frame



In [42]:
# Ouvrir la vidéo
video_path=('data/sample_video.mp4')
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print('Error: Could not open video.')
    exit()




while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (800, 600))
    frame = car_detection(frame)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
    
    

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 car, 373.0ms
Speed: 51.0ms preprocess, 373.0ms inference, 8.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 car, 457.6ms
Speed: 5.5ms preprocess, 457.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 cars, 377.9ms
Speed: 4.9ms preprocess, 377.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 353.7ms
Speed: 5.3ms preprocess, 353.7ms inference, 2.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 1 truck, 319.3ms
Speed: 5.0ms preprocess, 319.3ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 cars, 300.4ms
Speed: 8.8ms preprocess, 300.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 cars, 295.7ms
Speed: 7.2ms preprocess, 295.7ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 cars, 301.9ms
Speed: 4.3ms preprocess, 301.9ms inference, 0.0ms postprocess per image at shape (

In [5]:
#conversion de la vidéo en niveau de gris
cap = cv2.VideoCapture(video_path)
def grayscale(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return gray

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (800, 600))
    frame = grayscale(frame)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [33]:
#détection des lignes blanches

def white_line_detection(frame):
     # Convertir l'image en nuances de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Appliquer un flou pour réduire le bruit
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Détecter les lignes blanches en utilisant un seuil
    _, binary = cv2.threshold(blurred, 200, 255, cv2.THRESH_BINARY)  # Seuillage pour détecter les zones blanches
    # Détecter les contours
    edges = cv2.Canny(binary, 50, 150)  # Détection des bords avec Canny
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 150, maxLineGap=10)
    if lines is not None:
        #vérifier si la ligne est blanche
        
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    return frame

cap = cv2.VideoCapture(video_path)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (800, 600))
    frame = white_line_detection(frame)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [40]:
#grille de l'image correspondant à chaque pixel
def grid_image(frame):
    h, w = frame.shape[:2]
    grid = np.zeros_like(frame)  # Crée une image noire de même taille
    # Dessiner les lignes verticales
    for x in range(w):
        cv2.line(grid, (x, 0), (x, h), (255, 0, 0), 1)  # Bleu
    # Dessiner les lignes horizontales
    for y in range(h):
        cv2.line(grid, (0, y), (w, y), (0, 255, 0), 1)  # Vert
    # Combiner l'image originale avec la grille
    combined = cv2.addWeighted(frame, 0.8, grid, 0.2, 0)  # Fusionner avec un ratio (0.8 image, 0.2 grille)
    return combined

cap = cv2.VideoCapture(video_path)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (800, 600))
    frame = grid_image(frame)
    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()